In [0]:
dbutils.widgets.text("catalog", "fk_ce1366")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.read.table(f"{catalog}.bronze.products")

#casting
df = (
    df.filter(col("product_id").isNotNull())
    .withColumn("product_id", col("product_id").cast("string"))
    .withColumn("product_name", trim(initcap(col("product_name"))))
    .withColumn("brand", trim(initcap(col("brand"))))
    .withColumn("category_l1", trim(initcap(col("category.l1"))))
    .withColumn("category_l2", trim(initcap(col("category.l2"))))
    .withColumn("pack_size", trim(col("pack.size")).cast("int"))
    .withColumn("pack_unit", trim(initcap(col("pack.unit"))))
    .withColumn("mrp", col("mrp").cast("decimal(10,2)"))
    .withColumn("selling_price", col("selling_price").cast("decimal(10,2)"))
    .withColumn("is_active", col("is_active").cast("boolean"))
    .withColumn("last_changed_snapshot_date", to_date(col("snapshot_date")))
    .withColumn("tag", explode(col("tags"))).drop("tags")
    )

df = (
    df.drop("pack")
    .drop("category")
    .drop("_source_file")
    .drop("ingested_at")
    .drop("snapshot_date")
)

df = df.dropDuplicates()
df = df.dropna(how = "all")


df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.products")

In [0]:
%sql
select count(*) from fk_ce1366.silver.products_scd_t1

count(*)
96


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.products_scd_t1 (
    product_id STRING,
    product_name STRING,
    brand STRING,
    category_l1 STRING,
    category_l2 STRING,
    pack_size INT,
    pack_unit STRING,
    mrp DECIMAL(10,2),
    selling_price DECIMAL(10,2),
    is_active BOOLEAN,
    tag STRING,
    last_changed_snapshot_date Date
)
USING DELTA    
""")

DataFrame[]

In [0]:
spark.sql(f"""
ALTER TABLE {catalog}.silver.products_scd_t1
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);          
""")

DataFrame[]

In [0]:
spark.sql(f"""
MERGE INTO {catalog}.silver.products_scd_t1 AS t
USING {catalog}.silver.products AS s
ON t.product_id = s.product_id

WHEN MATCHED AND (
       NOT (t.product_name <=> s.product_name)
    OR NOT (t.brand <=> s.brand)
    OR NOT (t.category_l1 <=> s.category_l1)
    OR NOT (t.category_l2 <=> s.category_l2)
    OR NOT (t.pack_size <=> s.pack_size)
    OR NOT (t.pack_unit <=> s.pack_unit)
    OR NOT (t.mrp <=> s.mrp)
    OR NOT (t.selling_price <=> s.selling_price)
    OR NOT (t.is_active <=> s.is_active)
    OR NOT (t.tag <=> s.tag)
)
THEN UPDATE SET
    t.product_name = s.product_name,
    t.brand = s.brand,
    t.category_l1 = s.category_l1,
    t.category_l2 = s.category_l2,
    t.pack_size = s.pack_size,
    t.pack_unit = s.pack_unit,
    t.mrp = s.mrp,
    t.selling_price = s.selling_price,
    t.is_active = s.is_active,
    t.tag = s.tag,
    t.last_changed_snapshot_date = s.last_changed_snapshot_date

WHEN NOT MATCHED THEN
INSERT (
    product_id, product_name, brand,
    category_l1, category_l2, pack_size, pack_unit,
    mrp, selling_price, is_active, tag,
    last_changed_snapshot_date
)
VALUES (
    s.product_id, s.product_name, s.brand,
    s.category_l1, s.category_l2, s.pack_size, s.pack_unit,
    s.mrp, s.selling_price, s.is_active, s.tag,
    s.last_changed_snapshot_date
)
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from fk_ce1366.bronze.products

brand,category,is_active,mrp,pack,product_id,product_name,selling_price,snapshot_date,tags,_source_file,ingested_at
FreshKart Farms,"List(Fruits & Vegetables, Fresh Fruits)",true,60.0,"List(6, pcs)",P001,Banana Robusta,54.0,2026-03-01,"List(fresh, daily)",dbfs:/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-13T09:05:28.557Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Fruits)",true,180.0,"List(4, pcs)",P002,Apple Shimla,159.0,2026-03-01,List(fresh),dbfs:/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-13T09:05:28.557Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Fruits)",true,150.0,"List(3, pcs)",P003,Pomegranate,135.0,2026-03-01,List(fresh),dbfs:/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-13T09:05:28.557Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Vegetables)",true,45.0,"List(1, kg)",P004,Onion,39.0,2026-03-01,"List(fresh, daily, staple)",dbfs:/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-13T09:05:28.557Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Vegetables)",true,30.0,"List(500, g)",P005,Tomato Hybrid,26.0,2026-03-01,"List(fresh, daily)",dbfs:/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-13T09:05:28.557Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Vegetables)",true,40.0,"List(1, kg)",P006,Potato,35.0,2026-03-01,"List(fresh, staple)",dbfs:/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-13T09:05:28.557Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Vegetables)",true,15.0,"List(100, g)",P007,Coriander Leaves,12.0,2026-03-01,"List(fresh, daily)",dbfs:/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-13T09:05:28.557Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Vegetables)",true,12.0,"List(100, g)",P008,Green Chilli,10.0,2026-03-01,List(fresh),dbfs:/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-13T09:05:28.557Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Vegetables)",true,30.0,"List(250, g)",P009,Lemon,25.0,2026-03-01,List(fresh),dbfs:/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-13T09:05:28.557Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Vegetables)",true,28.0,"List(500, g)",P010,Cucumber,24.0,2026-03-01,List(fresh),dbfs:/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-13T09:05:28.557Z


In [0]:
%sql
select * from fk_ce1366.silver.products

brand,is_active,mrp,product_id,product_name,selling_price,category_l1,category_l2,pack_size,pack_unit,last_changed_snapshot_date,tag
Amul,true,95.00,P017,Amul Fresh Paneer,90.00,Dairy & Breakfast,Paneer & Curd,200,G,2026-03-01,dairy
Mother Dairy,true,50.00,P018,Mother Dairy Dahi,47.00,Dairy & Breakfast,Paneer & Curd,400,G,2026-03-01,daily
Fortune,true,165.00,P028,Fortune Sunflower Oil,149.00,Staples,Edible Oil,1,L,2026-03-01,cooking
Maggi,true,56.00,P045,Maggi 2-minute Noodles,54.00,Snacks & Beverages,Instant Food,280,G,2026-03-01,instant
Harvest Gold,true,45.00,P014,Harvest Gold White Bread,42.00,Dairy & Breakfast,Bread & Eggs,400,G,2026-03-01,bakery
Coca-cola,true,40.00,P036,Coca-cola,38.00,Snacks & Beverages,Soft Drinks,750,Ml,2026-03-01,beverage
Harpic,true,215.00,P048,Harpic Toilet Cleaner,189.00,Household,Cleaning,1,L,2026-03-01,household
Freshkart Farms,true,60.00,P001,Banana Robusta,54.00,Fruits & Vegetables,Fresh Fruits,6,Pcs,2026-03-01,daily
Tata,true,28.00,P030,Tata Salt,27.00,Staples,Salt & Sugar,1,Kg,2026-03-01,daily
Coca-cola,true,40.00,P036,Coca-cola,38.00,Snacks & Beverages,Soft Drinks,750,Ml,2026-03-01,impulse
